# Macro Surprise + VWAP Execution: First US Equity Screening

*A first attempt to test whether macroeconomic surprise days can create short-term tradeable patterns on selected US equities.*

This notebook is the readable summary of the project. The Python scripts stay as the backtesting engine. Here I only load the generated CSV outputs and figures, then explain what I tested, what the robustness checks mean, and where the result still needs work.

## Research Idea

I wanted to test whether macroeconomic surprises can help identify short-term opportunities on individual equities.

The idea is not just to buy good macro news and sell bad macro news. The goal is to combine a macro surprise signal with a more disciplined intraday execution rule based on VWAP.

For this US version, I should be careful with the wording. Many important US macro releases come out before the US equity open. So the current version is better interpreted as:

**macro calendar day + VWAP intraday**

rather than:

**pure minute-by-minute reaction to the release timestamp**

That distinction matters. If I show this to a trader, I do not want to pretend the backtest is measuring a clean release-second reaction when the entry is often at or after the equity open.

## Data Used

The study combines two datasets:

- US macroeconomic releases, with actual, estimate, and previous values when available.
- Historical surprise metrics by event type, used to normalize the surprise.
- Minute-level OHLCV stock data.
- Intraday VWAP computed from minute bars, with the VWAP reset each trading day.

Raw data is not included in this repo. The notebook only expects derived summary CSVs and report figures.

## Signal Construction

The macro surprise is built in three steps:

```text
surprise_raw = actual - estimate
surprise_z = (surprise_raw - historical_average_surprise) / historical_surprise_std
signal = surprise_z * direction
```

`direction = +1` when a higher macro value is treated as good.

`direction = -1` when a lower macro value is treated as good.

Events marked as `MIXED` are excluded because the sign is not clean enough.

The engine then tests two hypotheses:

- `DRIFT`: trade in the direction of the macro signal.
- `FADE`: trade against the macro signal.

## VWAP Execution Logic

The VWAP rule is used as an execution filter, not as the macro signal itself.

For a long trade, the model prefers entering when the price is below or close to VWAP. For a short trade, it prefers entering when the price is above or close to VWAP. If the price is not favorable, the model waits for a return toward VWAP within a defined window.

There are two important execution modes:

- `close_cross`: more optimistic, because it assumes the close crossing the condition is enough.
- `limit_touch`: more conservative, because it requires a cleaner touch/fill logic around VWAP.

VWAP is reset daily. That makes sense for intraday execution, but it also means the strategy can become partly a VWAP mean-reversion test if I am not careful. This is why the random tests and train/test split are important.

## Robustness Framework

The model does not keep a stock only because the full backtest looks profitable.

The robustness checks are:

- random side test
- random timestamp test
- time-shift placebo
- transaction cost sweep
- chronological train/test split
- walk-forward by year

The train/test split is especially important. The rule is selected on the first 70% of the period and then tested on the last 30%. If the rule looks good in the full sample but loses money on the test period, I reject it or treat it as fragile.

## Load Outputs

The notebook expects the generated files to be placed in:

```text
../reports/trader_report/csv_outputs/
```

It also looks for saved PNG charts in:

```text
../reports/trader_report/figures/
```

If a file is missing, the notebook prints a warning and keeps going.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    plt = None
    HAS_MATPLOTLIB = False

try:
    from IPython.display import display, Image, Markdown
except ModuleNotFoundError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image({self.filename})"

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "notebooks").exists() and (PROJECT_ROOT.parent / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_DIR = PROJECT_ROOT / "reports" / "trader_report" / "csv_outputs"
FIG_DIR = PROJECT_ROOT / "reports" / "trader_report" / "figures"
NOTES_DIR = PROJECT_ROOT / "reports" / "trader_report" / "notes"

FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTES_DIR.mkdir(parents=True, exist_ok=True)

expected_files = {
    "master": "single_country_master_summary.csv",
    "all_configs": "single_country_all_configs_summary.csv",
    "random_tests": "single_country_random_tests.csv",
    "train_test": "single_country_train_test.csv",
    "walk_forward": "single_country_walk_forward.csv",
}

loaded = {}
missing = []

for key, filename in expected_files.items():
    path = CSV_DIR / filename
    if path.exists():
        loaded[key] = pd.read_csv(path)
        print(f"Loaded {filename}: {loaded[key].shape[0]} rows, {loaded[key].shape[1]} columns")
    else:
        missing.append(filename)
        print(f"WARNING: missing {filename}")

print("\nCSV folder:", CSV_DIR)
print("Figure folder:", FIG_DIR)

if missing:
    display(Markdown("**Missing files:** " + ", ".join(f"`{m}`" for m in missing)))
else:
    display(Markdown("All expected CSV outputs were loaded."))

master = loaded.get("master")
all_configs = loaded.get("all_configs")
random_tests = loaded.get("random_tests")
train_test = loaded.get("train_test")
walk_forward = loaded.get("walk_forward")

## Ticker Checks

Ticker naming is a small thing, but it can create confusion in a review. I especially want to catch two cases:

- a ticker displayed as `LONDON-STRATEGIC-EDGE`, which is probably a filename parsing issue
- duplicated tickers, such as the same raw CSV being present twice

The code below does not change the backtest result. It only flags presentation issues.

In [ ]:
def extract_ticker_from_filename(name):
    base = Path(name).stem
    base = re.sub(r"\s*\(\d+\)$", "", base)
    m = re.match(r"([A-Za-z]+)_dataset", base)
    if m:
        return m.group(1).upper()
    first = re.split(r"[_\-\s]", base)[0]
    return first.upper() if first else base.upper()

notes = []

if master is not None and "ticker" in master.columns:
    tickers = master["ticker"].astype(str)
    bad = tickers.str.upper().eq("LONDON-STRATEGIC-EDGE")
    if bad.any():
        notes.append("`LONDON-STRATEGIC-EDGE` appears as a ticker in the summary. I would treat this as a presentation issue, likely caused by filename parsing. If the source file is the LMB file, show it as LMB in tables and keep the raw result unchanged.")

    dupes = tickers[tickers.duplicated(keep=False)].sort_values().unique().tolist()
    if dupes:
        notes.append("Duplicate tickers appear in the master summary: " + ", ".join(f"`{d}`" for d in dupes))

raw_dir = PROJECT_ROOT / "US-Actions"
if raw_dir.exists():
    raw_files = sorted(raw_dir.glob("*.csv"))
    raw_tickers = [extract_ticker_from_filename(p.name) for p in raw_files]
    raw_dupes = sorted({t for t in raw_tickers if raw_tickers.count(t) > 1})
    if raw_dupes:
        notes.append("Duplicate-looking raw input files are present locally: " + ", ".join(f"`{d}`" for d in raw_dupes) + ". I would clean these before a final rerun.")
    if any(t == "LMB" for t in raw_tickers):
        notes.append("An `LMB` raw source file is present locally. If a result shows `LONDON-STRATEGIC-EDGE`, I would display it as `LMB (?)` and mention the filename issue.")

if notes:
    display(Markdown("\n\n".join("- " + n for n in notes)))
else:
    display(Markdown("No ticker naming issues detected from the loaded summaries or local raw filenames."))

## Results Overview

This section uses `single_country_master_summary.csv`. It gives the high-level picture: how many stocks were tested, how the classifications split, which names rank highest by confidence score, and which names were rejected.

In [ ]:
def display_master_overview(master_df):
    if master_df is None:
        display(Markdown("`single_country_master_summary.csv` is missing, so the overview table cannot be built yet."))
        return None

    df = master_df.copy()

    if "ticker" in df.columns:
        bad = df["ticker"].astype(str).str.upper().eq("LONDON-STRATEGIC-EDGE")
        if bad.any():
            df.loc[bad, "ticker"] = "LMB (?)"

    n = len(df)
    display(Markdown(f"**Stocks in master summary:** {n}"))

    if "classification" in df.columns:
        counts = df["classification"].fillna("MISSING").value_counts().rename_axis("classification").reset_index(name="count")
        display(counts)

    table_cols = [
        "ticker",
        "classification",
        "confidence_score",
        "best_hypothese",
        "best_vwap_mode",
        "best_horizon",
        "best_ret_moy_pct",
        "test_ret_moy_pct",
        "random_side_percentile",
        "random_ts_percentile",
        "cost_break_even_bps",
        "main_warning",
    ]
    present_cols = [c for c in table_cols if c in df.columns]

    sort_cols = [c for c in ["confidence_score", "test_ret_moy_pct", "best_ret_moy_pct"] if c in df.columns]
    if sort_cols:
        ranked = df.sort_values(sort_cols, ascending=False)
    else:
        ranked = df

    display(Markdown("**Top candidates by confidence score**"))
    display(ranked[present_cols].head(12) if present_cols else ranked.head(12))

    if "classification" in df.columns:
        rejected_mask = df["classification"].astype(str).str.contains("D_REJECT|INSUFFICIENT", case=False, na=False)
        rejected = df.loc[rejected_mask]
        display(Markdown("**Rejected or insufficient-data names**"))
        if rejected.empty:
            display(Markdown("No rejected or insufficient-data rows in the loaded master summary."))
        else:
            display(rejected[present_cols].head(30) if present_cols else rejected.head(30))

    return df

master_view = display_master_overview(master)

## Existing Figures Or Fallback Charts

If PNG figures already exist, I display them directly. If not, the notebook tries to generate simple charts from the CSV outputs.

The fallback charts are intentionally plain: readable titles, clear labels, and no flashy styling.

In [ ]:
def save_current_fig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=140, bbox_inches="tight")
    plt.show()
    print("Saved", path)
    return path

existing_pngs = sorted(FIG_DIR.glob("*.png"))

if existing_pngs:
    display(Markdown("**Saved report figures found:**"))
    for path in existing_pngs:
        display(Markdown(f"`{path.name}`"))
        display(Image(filename=str(path)))
elif not HAS_MATPLOTLIB:
    display(Markdown("No saved PNG figures found, and `matplotlib` is not installed in this environment. Install the requirements to generate fallback charts."))
else:
    display(Markdown("No saved PNG figures found. I will generate fallback charts where the data allows it."))

    generated = []

    if master_view is not None and "classification" in master_view.columns:
        counts = master_view["classification"].fillna("MISSING").value_counts()
        fig, ax = plt.subplots(figsize=(8, 4.5))
        counts.plot(kind="bar", ax=ax, color="#4c78a8")
        ax.set_title("Classification count")
        ax.set_xlabel("Classification")
        ax.set_ylabel("Number of stocks")
        ax.tick_params(axis="x", rotation=35)
        generated.append(save_current_fig("01_classification_count.png"))

    if master_view is not None and {"ticker", "confidence_score"}.issubset(master_view.columns):
        top = master_view.sort_values("confidence_score", ascending=False).head(10).copy()
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top["ticker"].astype(str), top["confidence_score"], color="#59a14f")
        ax.invert_yaxis()
        ax.set_title("Top 10 assets by confidence score")
        ax.set_xlabel("Confidence score")
        ax.set_ylabel("Ticker")
        generated.append(save_current_fig("02_top_10_confidence_score.png"))

    if master_view is not None and {"ticker", "best_ret_moy_pct", "test_ret_moy_pct"}.issubset(master_view.columns):
        plot_df = master_view.dropna(subset=["best_ret_moy_pct", "test_ret_moy_pct"]).copy()
        if not plot_df.empty:
            fig, ax = plt.subplots(figsize=(7.5, 5))
            ax.scatter(plot_df["best_ret_moy_pct"], plot_df["test_ret_moy_pct"], color="#4c78a8", alpha=0.8)
            ax.axhline(0, color="#666666", linewidth=1)
            ax.axvline(0, color="#666666", linewidth=1)
            ann = plot_df.sort_values("confidence_score", ascending=False).head(10) if "confidence_score" in plot_df else plot_df.head(10)
            for _, row in ann.iterrows():
                ax.annotate(str(row["ticker"]), (row["best_ret_moy_pct"], row["test_ret_moy_pct"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
            ax.set_title("Full-period edge vs chronological test edge")
            ax.set_xlabel("Best full-sample average return (%)")
            ax.set_ylabel("Test-period average return (%)")
            generated.append(save_current_fig("03_full_period_vs_test_edge.png"))

    if master_view is not None and {"ticker", "random_side_percentile", "cost_break_even_bps"}.issubset(master_view.columns):
        plot_df = master_view.dropna(subset=["random_side_percentile", "cost_break_even_bps"]).copy()
        if not plot_df.empty:
            fig, ax = plt.subplots(figsize=(7.5, 5))
            ax.scatter(plot_df["random_side_percentile"], plot_df["cost_break_even_bps"], color="#f28e2b", alpha=0.8)
            ann = plot_df.sort_values("confidence_score", ascending=False).head(10) if "confidence_score" in plot_df else plot_df.head(10)
            for _, row in ann.iterrows():
                ax.annotate(str(row["ticker"]), (row["random_side_percentile"], row["cost_break_even_bps"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
            ax.set_title("Random-side percentile vs cost break-even")
            ax.set_xlabel("Random side percentile")
            ax.set_ylabel("Cost break-even (bps)")
            generated.append(save_current_fig("04_random_side_vs_cost_break_even.png"))

    wf_ratio = None
    if master_view is not None and {"ticker", "walk_forward_positive_ratio"}.issubset(master_view.columns):
        wf_ratio = master_view[["ticker", "walk_forward_positive_ratio"]].dropna().copy()
    elif walk_forward is not None and "ticker" in walk_forward.columns:
        ret_col = next((c for c in ["ret_moy_pct", "ret_moy_%", "best_ret_moy_pct"] if c in walk_forward.columns), None)
        if ret_col:
            wf_ratio = (walk_forward.assign(_positive=walk_forward[ret_col] > 0)
                        .groupby("ticker", as_index=False)["_positive"].mean()
                        .rename(columns={"_positive": "walk_forward_positive_ratio"}))

    if wf_ratio is not None and not wf_ratio.empty:
        top_wf = wf_ratio.sort_values("walk_forward_positive_ratio", ascending=False).head(15)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top_wf["ticker"].astype(str), top_wf["walk_forward_positive_ratio"], color="#79706e")
        ax.invert_yaxis()
        ax.set_title("Walk-forward positive ratio by asset")
        ax.set_xlabel("Share of positive walk-forward periods")
        ax.set_ylabel("Ticker")
        generated.append(save_current_fig("05_walk_forward_positive_ratio.png"))

    if not generated:
        display(Markdown("No fallback charts could be generated because the required CSV data is missing."))

## Interpretation

This cell writes a short interpretation from the loaded master summary, so the examples come from the data instead of being hardcoded.

In [ ]:
def names_from(df, mask, limit=5):
    if df is None or "ticker" not in df.columns:
        return []
    sub = df.loc[mask].copy()
    if "confidence_score" in sub.columns:
        sub = sub.sort_values("confidence_score", ascending=False)
    return sub["ticker"].astype(str).head(limit).tolist()

if master_view is None:
    display(Markdown("The interpretation will update once `single_country_master_summary.csv` is available."))
else:
    df = master_view.copy()
    cls = df["classification"].astype(str) if "classification" in df.columns else pd.Series([""] * len(df), index=df.index)
    a_names = names_from(df, cls.str.startswith("A"))
    rejected_names = names_from(df, cls.str.contains("D_REJECT|INSUFFICIENT", case=False, na=False))
    fragile_names = names_from(df, cls.str.startswith("B") | cls.str.startswith("C"))

    msg = f"The first US batch in the loaded summary contains {len(df)} names. "
    if a_names:
        msg += "The cleaner A-class candidates are " + ", ".join(a_names) + ". "
    else:
        msg += "There are no A-class candidates in the loaded summary. "
    if fragile_names:
        msg += "A larger group looks more fragile or more VWAP-driven, for example " + ", ".join(fragile_names) + ". "
    if rejected_names:
        msg += "Rejected or insufficient-data names include " + ", ".join(rejected_names) + ". "
    msg += "The main thing I care about is not just the full-sample return. If the train-selected rule fails on the later test period, I do not want to treat the full-sample result as reliable."

    display(Markdown(msg))

## Why Profitable Names Can Be Rejected

A stock can have a positive full-sample backtest and still be rejected.

This happens when the model finds a profitable configuration using the full dataset, but the configuration selected on the earlier period does not work on the later test period. In that case, the result may be overfit, regime-dependent, or mostly caused by the VWAP filter rather than the macro signal.

The table below looks for stocks where:

```text
best_ret_moy_pct > 0
test_ret_moy_pct <= 0
```

In [ ]:
if master_view is None:
    display(Markdown("Cannot build this table yet because the master summary CSV is missing."))
elif {"best_ret_moy_pct", "test_ret_moy_pct"}.issubset(master_view.columns):
    rejected_profitable = master_view[
        (pd.to_numeric(master_view["best_ret_moy_pct"], errors="coerce") > 0)
        & (pd.to_numeric(master_view["test_ret_moy_pct"], errors="coerce") <= 0)
    ].copy()

    cols = ["ticker", "best_ret_moy_pct", "train_ret_moy_pct", "test_ret_moy_pct", "classification", "main_warning"]
    cols = [c for c in cols if c in rejected_profitable.columns]

    if rejected_profitable.empty:
        display(Markdown("No loaded rows match this filter."))
    else:
        display(rejected_profitable[cols].sort_values("best_ret_moy_pct", ascending=False))
else:
    display(Markdown("The required columns are not present in the loaded master summary."))

## Main Warning

The main warning is that the current US setup has many `PRE_OPEN_SAME_DAY` entries.

Because of that, I should not present this as a pure high-frequency reaction to macro releases. The more accurate interpretation is that I am testing whether macro release days create an intraday environment where a VWAP-based execution rule can capture directional or contrarian patterns.

That is still interesting, but it is a different claim. Before treating anything as a live trading idea, I would want cleaner event timestamps, more simulations, and more conservative execution assumptions.

## Next Steps

- Rerun the best candidates with more random simulations.
- Clean ticker naming and remove duplicate raw files before the final batch.
- Test `limit_touch` more deeply, because it is closer to realistic execution.
- Improve event timestamp handling, especially for pre-open US releases.
- Test a larger but controlled universe.
- Build a small paper-trading version only after the candidates survive the stricter checks.